In [1]:
from h5dataset import h5set
import os
import torch 
import torch.nn as nn
from torch.utils.data import DataLoader, ConcatDataset
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
print(f'Selected device: {device}')

Selected device: cuda


In [2]:
dir_path = "/mnt/data/train_test_val"
h5_path = "../data/H1_rechunked.h5"
dir_ = os.listdir(dir_path)
if len(dir_)==0:
    noise = h5set(path=h5_path, dataset='noise')
    train, val, test = torch.utils.data.random_split(noise, [225000, 75000, 25000])
    injection = h5set(path=h5_path, dataset='injection')
    test = ConcatDataset([test, injection])
    torch.save(train, "/mnt/data/train_test_val/train.pt") #just saves indices
    torch.save(val, "/mnt/data/train_test_val/val.pt")
    torch.save(test, "/mnt/data/train_test_val/test.pt")
else:
    train = torch.load("/mnt/data/train_test_val/train.pt", weights_only=False)
    val = torch.load("/mnt/data/train_test_val/val.pt", weights_only=False)
    test = torch.load("/mnt/data/train_test_val/test.pt", weights_only=False)


training = DataLoader(train,
                batch_size=1,
                shuffle=True,
                num_workers=0,
                #persistent_workers=True,
                drop_last=True,
                )


In [2]:
# local run
data = h5set('/mnt/eph/data/l1500mb_c.h5', 100, 30)
training = DataLoader(data, batch_size=1, num_workers=os.cpu_count())

In [58]:
class Encoder_Moreno(nn.Module):
    def __init__(self, num_feat, exp_dim, compr_dim, num_layers, v=False):
        super().__init__()
        ## Useful quantities
        self.v = v
        self.El1 = nn.LSTM(input_size=num_feat, 
                           hidden_size=exp_dim,
                           num_layers=num_layers,
                           batch_first=True)
        self.El2 = nn.LSTM(input_size=exp_dim,
                           hidden_size=compr_dim, 
                           num_layers=num_layers,
                           batch_first=True)

    
    def forward(self, item):
        sq_len = item.shape[1]  # to be changed if batch_first=False
        if self.v: 
            print(item.shape, "Input shape")
            item, _ = self.El1(item)
            print(item.shape, "1st encoder layer output shape")
            item, _ = self.El2(item)
            print(item.shape, "2nd encoder layer output shape")
            item = item[:,-1,:]
            print(item.shape, "return sequence= False analog")
            item = item.repeat(1, sq_len, 1)
            print(item.shape, "repeat vector 100x")
            return item
        else:
            item, h_c = self.El1(item)
            item, h_c = self.El2(item)
            item = item[:,-1,:]
            item = item.repeat(1, sq_len,1)
            return item



class Decoder_Moreno(nn.Module):
    def __init__(self, num_feat, exp_dim, compr_dim, num_layers, v=False):
        super().__init__()
        self.v = v
        self.Dl1 = nn.LSTM(input_size=compr_dim, 
                           hidden_size=compr_dim,
                           num_layers=num_layers,
                           batch_first=True)
        self.Dl2 = nn.LSTM(input_size=compr_dim,
                           hidden_size=exp_dim,
                           num_layers=num_layers,
                           batch_first=True)
        self.TimeDistributed = nn.Conv1d(exp_dim,
                                        num_feat,
                                        kernel_size=1)

        
    def forward(self, item):
        if self.v: 
            item, h_c = self.Dl1(item)
            print(item.shape, "1st decoder layer output shape")
            item, h_c = self.Dl2(item)
            print(item.shape, "2nd decoder layer output shape")
            item = torch.movedim(item, 1,2)
            print(item.shape, "move dim shape")
            item = self.TimeDistributed(item)
            print(item.shape, "conv1d shape (time distributed)")
            return item
        else:
            item, _ = self.Dl1(item)
            item, _ = self.Dl2(item)
            item = torch.movedim(item, 1,2)
            item = self.TimeDistributed(item)
            return item



class AEric(nn.Module):
    def __init__(self, num_feat, exp_dim, compr_dim, num_layers, v=False):
        super().__init__()
        self.Encoder = Encoder_Moreno(num_feat, exp_dim, compr_dim, num_layers, v=v)
        self.Decoder = Decoder_Moreno(num_feat, exp_dim, compr_dim, num_layers, v=v)

    
    def forward(self, item):
        item = self.Encoder(item)
        item = self.Decoder(item)
        return item
        # encoded = self.Encoder(item)
        # decoded = self.Decoder(encoded)
        # return decoded

In [59]:
AE = AEric(num_feat=1,
            exp_dim=32,
            compr_dim=8,
            num_layers=3,
            v=True)
AE.to(device)
loss_func = torch.nn.MSELoss()
lr = 5e-3
optim = torch.optim.Adam(AE.parameters(),
                        lr=lr,
                        weight_decay=1e-5
                        )



In [43]:
def traingio(model, device, dataloader, loss_fn, optim):
    model.train()
    epoch_loss = 0
    for batch_data in dataloader:
        optim.zero_grad()
        # print(batch_data.element_size() * batch_data.nelement())
        batch_data = batch_data.reshape(-1, 100, 1).to(device)
        print(batch_data.shape)
        output = model(batch_data)
        loss = loss_fn(output, batch_data)
        loss.backward()
        optim.step()
        # loss = np.sqrt(loss.item()) # if need
        epoch_loss += loss
    return epoch_loss / len(dataloader)

In [5]:
def train_epoch(ae, device, dataloader,timestep, loss_fn, optim):
    ae.train()
    losses = []
    for batch_data in dataloader:
        num_time_steps = batch_data.shape[1]
        remainder = num_time_steps % timestep
        batch_data = batch_data[:, :-remainder]
        for sample_idx in range(batch_data.shape[0]):
            sequence = batch_data[sample_idx, :]
            segments = sequence.reshape(-1, timestep, 1)
            for segment_idx in range(segments.shape[0]):
                current_window = segments[segment_idx, :, :]
                c_w = current_window.unsqueeze(0)
                c_w = c_w.to(device)
                ae_output = ae(c_w)
                loss = loss_fn(ae_output,c_w)
                print(loss)
                optim.zero_grad()
                loss.backward()
                optim.step()

                losses.append(loss.detach().cpu().numpy())
    losses = np.mean(losses)
    return losses

In [44]:
num_epochs = 10
losses = []
for epoch in range(num_epochs):
    ### Training (use the training function)
    train_loss = traingio(
        model=AE,
        device=device,
        dataloader=training,
        loss_fn=loss_func,
        optim=optim)
    print(f'TRAIN - EPOCH {epoch+1}/{num_epochs} - loss: {train_loss}')
    losses.append(train_loss)

torch.Size([543, 100, 1])


/home/teo/miniconda3/envs/killgg/lib/python3.12/site-packages/torch/nn/modules/loss.py:634: UserWarning: Using a target size (torch.Size([543, 100, 1])) that is different to the input size (torch.Size([1, 1, 54300])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


OutOfMemoryError: CUDA out of memory. Tried to allocate 10.98 GiB. GPU 0 has a total capacity of 5.60 GiB of which 4.27 GiB is free. Including non-PyTorch memory, this process has 1.33 GiB memory in use. Of the allocated memory 782.57 MiB is allocated by PyTorch, and 481.43 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [96]:
class EricGio(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.El1 = nn.LSTM(input_size=1, hidden_size=32, num_layers=3, batch_first=True)  # Send whole output (corresponds to 100x32 in paper)
  
        self.El2 = nn.LSTM(input_size=32, hidden_size=8, num_layers=3, batch_first=True)  # Send last output (corresponds to whole[-1], has 1 value for each layer (3)) I guess the paper takes only the last of these x[-1]
        
        self.Dl1 = nn.LSTM(input_size=8, hidden_size=8, num_layers=3, batch_first=True)
        
        self.Dl2 = nn.LSTM(input_size=8, hidden_size=32, num_layers=3, batch_first=True)
        
        self.TimeDistributed = nn.Conv1d(in_channels=32, out_channels=1, kernel_size=1)   # Not sure it corresponds to TimeDistributed(Dense), also stoopid LSTM output are weird, and does not work with Conv1d. Perhaps we ought to do manual labor

    def forward(self, x):
        # Encoding
        print(f'Input {x.shape=}')
        x, _ = self.El1(x)  
        print(f"First LSTM out {x.shape=}")
        
        ## gg El2
        x, _ = self.El2(x)  ## gg method
        print(f"Second LSTM out {x[:,-1,:].shape=}")

        ## gio El2
        # _, (x, _) = self.El2(x)  
        # print(f"Second LSTM out {x[-1].shape=}")

        # Repeating
        # x = x[-1].unsqueeze(1).repeat(1, 100, 1)  # Tensor.unsqueeze(x) adds a dimension to x position, to have batch dim back.  ## gio method
        x = x[:, -1, :].repeat(1, 100, 1)  ## gg method
        print(f"Repeated {x.shape=}")

        # Decoding
        print("Decoding")
        x, _ = self.Dl1(x)
        print(f"First LSTM out {x.shape=}")
        x, _ = self.Dl2(x)
        print(f"Second LSTM out {x.shape=}")
        x = torch.movedim(x, 1, 2)
        print(f"3D transposed {x.shape=}")
        x = self.TimeDistributed(x)
        print(f'Convoluted {x.shape=}')
        x = torch.movedim(x, 1, 2)
        print(f'Back to original dim {x.shape=}')
        return x

In [97]:
ehi = EricGio()
ehi.to(device)
with torch.no_grad():
    ehi.eval()
    ehi(torch.randn(543, 100, 1).to(device))

Input x.shape=torch.Size([543, 100, 1])
First LSTM out x.shape=torch.Size([543, 100, 32])
Second LSTM out x[:,-1,:].shape=torch.Size([543, 8])
Repeated x.shape=torch.Size([543, 100, 8])
Decoding
First LSTM out x.shape=torch.Size([543, 100, 8])
Second LSTM out x.shape=torch.Size([543, 100, 32])
3D transposed x.shape=torch.Size([543, 32, 100])
Convoluted x.shape=torch.Size([543, 1, 100])
Back to original dim x.shape=torch.Size([543, 100, 1])


In [93]:
ehi = EricGio()
ehi.to(device)
with torch.no_grad():
    ehi.eval()
    ehi(torch.randn(543, 100, 1).to(device))

Input x.shape=torch.Size([543, 100, 1])
First LSTM out x.shape=torch.Size([543, 100, 32])
Second LSTM out x[:,-1,:].shape=torch.Size([543, 8])
Repeated x.shape=torch.Size([1, 54300, 8])
Decoding
First LSTM out x.shape=torch.Size([1, 54300, 8])
Second LSTM out x.shape=torch.Size([1, 54300, 32])
3D transposed x.shape=torch.Size([1, 32, 54300])
Convoluted x.shape=torch.Size([1, 1, 54300])
Back to original dim x.shape=torch.Size([1, 54300, 1])
